# Completed-play reconstruction and history audit

Corrective exploratory amendment after the original within-play leakage discovery. Archived results are retained; 2025 feeds remain closed. This notebook inspects checksum-verified completed runs generated by the documented scripts. It does not replace independent raw-response and history checks.

In [1]:
from scripts.artifact_runs import resolve_run
from scripts.recorded_attempt_baseline import ROOT
import json
import pandas as pd
rebuilt=resolve_run(ROOT/'results/grouped_event_rebuild',ROOT/'data/processed/grouped_event_rebuild')
histories=resolve_run(ROOT/'results/grouped_histories',ROOT/'data/processed/grouped_histories')
print('Reconstruction:',rebuilt.results.name)
print('Histories:',histories.results.name)
assert not (ROOT/'data/historical_feed_manifest_2025.csv').exists()

Reconstruction: 16aff9b562484e979ba80f50f073ac77
Histories: 897435bbc46547bd92d05fde705d8f89


In [2]:
print(pd.read_csv(rebuilt.results/'summary.csv').to_string(index=False))
print(pd.read_csv(histories.results/'sample_flow.csv').to_string(index=False))

 season  opportunities  recorded_attempt_windows  recorded_pitcher_pickoff_windows  terminal_windows  attempts_with_following_pitch  grouped_outcome_records
   2022         134972                      2272                             11525               375                           2212                     5415
   2023         135536                      2875                              7479               502                           2809                     6503
   2024         133103                      2994                              7635               527                           2919                     6612
 season  windows  games  attempts  pitcher_pickoffs  terminal
   2022   134972   2430      2272             11525       375
   2023   135536   2430      2875              7479       502
   2024   133103   2429      2994              7635       527


In [3]:
checks=pd.read_csv(histories.results/'response_boundary_checks.csv')
print(checks.groupby('season')[['windows','response_actions','explicit_links','attempts']].sum().to_string())
assert checks.game_pk.is_unique and len(checks)==7289
identity=pd.read_csv(histories.results/'identity_changes.csv')
identity['additional_exclusions']=identity.prior_history_eligible-identity.corrected_history_eligible
print(identity.groupby('season')[['changed_pitcher_rows','additional_exclusions']].sum().to_string())

        windows  response_actions  explicit_links  attempts
season                                                     
2022     134972             16014            2381      2272
2023     135536             12166            3138      2875
2024     133103             12608            3212      2994
        changed_pitcher_rows  additional_exclusions
season                                             
2022                    1239                   1754
2023                    1267                   1801
2024                    1248                   1836


In [4]:
provenance=json.loads((histories.results/'provenance.json').read_text())
print('Independent history equalities:',provenance['independent_history_equalities'])
print('Reference games:',len(provenance['independent_games']))
print('Environment:',provenance['environment'])
assert provenance['independent_history_equalities']>0

Independent history equalities: 5562
Reference games: 35
Environment: {'python': '3.14.4', 'pandas': '3.0.2', 'numpy': '2.4.4', 'sklearn': '1.8.0'}


The checks establish internal agreement with the cached public feeds and declared chronology rules. They do not establish that every real action is recorded, that event timestamps measure release times, that the work is novel, or that predictive associations are causal. Conditional-success semantics and independent video review remain separate gates.